# Storage Mapper — Missing Tank Pilot (Colab GPU · DOTA Aerial OBB)

Uses **YOLO26-OBB / YOLO11-OBB** (pretrained on DOTA where `storage tank` is a native top-down class) with:
- **OSM Exclusion Masking:** ignores mapped `man_made=clarifier`, `water=wastewater`, `silo`, `water_tower`, `gasometer`, `reservoir_covered`, and `landuse=basin`.
- **Automatic Off-Nadir Alignment:** estimates each farm's ground-base to roof shift, performs 1-to-1 matching, and suppresses double-detections on tank cylinder bodies/bases.

**How to run:**
1. Select **Runtime -> Change runtime type -> T4 GPU** (free tier).
2. Click **Runtime -> Run all**.

**Plot colors:**
- **Green solid:** Already mapped OSM `man_made=storage_tank` (ground base)
- **Green dotted:** Predicted roof position when imagery is off-nadir
- **Cyan:** Excluded OSM features (clarifiers, wastewater basins, silos, etc.)
- **Red dashed:** Unmapped storage tank candidates


In [ ]:
#@title 1. Pilot Settings
MODEL_NAME = 'yolo26x-obb.pt' #@param ['yolo26x-obb.pt', 'yolo11x-obb.pt', 'yolo26l-obb.pt', 'yolo11l-obb.pt']
COUNTRY_ISO2 = 'IQ' #@param ['IQ', 'SY', 'IR', 'TR', 'KW', 'SA', 'JO', 'AE']
PILOT_LIMIT = 5 #@param {type:'integer'}
MIN_FARM_TANKS = 4 #@param {type:'integer'}
MAX_FARM_TANKS = 25 #@param {type:'integer'}
CONF_THRESH = 0.35 #@param {type:'slider', min:0.15, max:0.85, step:0.05}
ZOOM = 18 #@param [17, 18] {type:'raw'}
BUFFER_M = 200 #@param {type:'integer'}
MAX_TILES_PER_FARM = 150 #@param {type:'integer'}
DOWNLOAD_GEOJSON = False #@param {type:'boolean'}

print(f'Config ready -> model={MODEL_NAME} | country={COUNTRY_ISO2} | pilot_farms={PILOT_LIMIT} | conf={CONF_THRESH} | zoom={ZOOM}')


In [ ]:
#@title 2. Install Ultralytics & Load Pretrained DOTA Aerial Model
%pip install -q ultralytics

import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU active: {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU (switch Runtime -> Change runtime type -> T4 GPU for 10x speed).')

model = YOLO(MODEL_NAME)
tank_ids = [k for k, v in model.names.items() if 'storage' in v.lower() and 'tank' in v.lower()]
if not tank_ids:
    raise RuntimeError(f'Could not locate storage tank class in model names: {model.names}')
TANK_CLS_ID = tank_ids[0]
print(f'Loaded {MODEL_NAME} | target class {TANK_CLS_ID}: {model.names[TANK_CLS_ID]}')


In [ ]:
#@title 3. Fetch OSM Storage Tanks, Clarifier/Silo Exclusions & Cluster Into Farms
import math, json, time, requests
import numpy as np
from scipy.spatial import cKDTree

OVERPASS_ENDPOINTS = [
    'https://maps.mail.ru/osm/tools/overpass/api/interpreter',
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.private.coffee/api/interpreter',
]

HEADERS = {
    'User-Agent': 'StorageMapper-Pilot/2.4 (+https://github.com/rawazrauf/storage_mapper)',
    'Accept': 'application/json',
}

QUERY = f'''[out:json][timeout:180];
area['ISO3166-1'='{COUNTRY_ISO2}'][admin_level=2]->.searchArea;
(
  way['man_made'='storage_tank'](area.searchArea);
  nwr['man_made'='clarifier'](area.searchArea);
  nwr['water'='wastewater'](area.searchArea);
  way['man_made'='silo'](area.searchArea);
  way['man_made'='water_tower'](area.searchArea);
  way['man_made'='gasometer'](area.searchArea);
  way['man_made'='reservoir_covered'](area.searchArea);
  way['landuse'='basin'](area.searchArea);
);
out geom tags;'''

def fetch_osm_features():
    for url in OVERPASS_ENDPOINTS:
        host = url.split('/')[2]
        for attempt in range(2):
            try:
                t0 = time.time()
                r = requests.post(url, data={'data': QUERY}, headers=HEADERS, timeout=180)
                if r.status_code == 429 and attempt == 0:
                    wait = min(int(r.headers.get('Retry-After', 8)), 15)
                    print(f'Rate-limited on {host} (429), retrying in {wait}s...')
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                data = r.json()
                if 'elements' in data:
                    n_el = len(data['elements'])
                    print(f'Fetched {n_el} elements from {host} in {time.time()-t0:.1f}s')
                    return data['elements']
            except Exception as e:
                print(f'Endpoint failed ({host}): {e}')
                break
    raise RuntimeError('All Overpass endpoints failed. Re-run this cell in a moment.')

def tank_metrics(geom):
    n = len(geom)
    lats = [p['lat'] for p in geom]
    lons = [p['lon'] for p in geom]
    lat_avg = sum(lats) / n
    lon_avg = sum(lons) / n
    cos_lat = math.cos(math.radians(lat_avg))
    area, perim = 0.0, 0.0
    for i in range(n - 1):
        x0, y0 = geom[i]['lon'], geom[i]['lat']
        x1, y1 = geom[i + 1]['lon'], geom[i + 1]['lat']
        area += x0 * y1 - x1 * y0
        dx = (x1 - x0) * cos_lat * 111320.0
        dy = (y1 - y0) * 111320.0
        perim += math.hypot(dx, dy)
    area = abs(area) / 2.0 * (111320.0 ** 2) * cos_lat
    circ = (4 * math.pi * area) / (perim ** 2) if perim > 0 else 0.0
    diam = 2.0 * math.sqrt(area / math.pi) if area > 0 else 0.0
    return lat_avg, lon_avg, circ, diam

def add_excl_ring(geom, label, target_list):
    if len(geom) < 3:
        return
    lats = [p['lat'] for p in geom]
    lons = [p['lon'] for p in geom]
    target_list.append({
        'kind': 'way',
        'label': label,
        'geom': geom,
        'min_lat': min(lats), 'max_lat': max(lats),
        'min_lon': min(lons), 'max_lon': max(lons),
    })

raw = fetch_osm_features()
all_osm_tanks = []
excluded_osm = []
tanks = []

for el in raw:
    tags = el.get('tags', {})
    etype = el.get('type')
    if etype == 'node':
        excluded_osm.append({
            'kind': 'node',
            'label': tags.get('man_made') or tags.get('water', 'excluded'),
            'lat': el['lat'], 'lon': el['lon'],
            'min_lat': el['lat'], 'max_lat': el['lat'],
            'min_lon': el['lon'], 'max_lon': el['lon'],
        })
        continue

    if etype == 'relation':
        label = tags.get('man_made') or tags.get('water', 'excluded')
        for mem in el.get('members', []):
            if mem.get('role', 'outer') in ('outer', '') and 'geometry' in mem:
                add_excl_ring(mem['geometry'], label, excluded_osm)
        continue

    geom = el.get('geometry', [])
    if len(geom) < 3:
        continue

    if tags.get('man_made') == 'storage_tank':
        lat, lon, circ, diam = tank_metrics(geom)
        rec = {'id': el['id'], 'lat': lat, 'lon': lon, 'diam_m': max(diam, 2.0), 'geom': geom}
        all_osm_tanks.append(rec)
        if circ >= 0.75 and diam >= 6.0:
            tanks.append(rec)
    else:
        label = tags.get('man_made') or tags.get('landuse') or tags.get('water', 'excluded')
        add_excl_ring(geom, label, excluded_osm)

mid_lat = float(np.mean([t['lat'] for t in tanks]))
cos_mid = math.cos(math.radians(mid_lat))
pts_m = np.array([[t['lat'] * 111320.0, t['lon'] * 111320.0 * cos_mid] for t in tanks])
tree = cKDTree(pts_m)
parent = list(range(len(tanks)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

for i, j in tree.query_pairs(r=500.0):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[ri] = rj

groups = {}
for idx, t in enumerate(tanks):
    groups.setdefault(find(idx), []).append(t)

clusters = sorted(groups.values(), key=len, reverse=True)
eligible = [c for c in clusters if MIN_FARM_TANKS <= len(c) <= MAX_FARM_TANKS]
print(f'OSM storage_tanks: {len(all_osm_tanks)} | Exclusions (clarifiers/wastewater/silos/basins): {len(excluded_osm)}')
print(f'Circular tanks (>=6m): {len(tanks)} across {len(clusters)} clusters')
print(f'Eligible farms ({MIN_FARM_TANKS}-{MAX_FARM_TANKS} tanks): {len(eligible)} | Pilot will scan: {min(PILOT_LIMIT, len(eligible)) if PILOT_LIMIT > 0 else len(eligible)}')


In [ ]:
#@title 4. Run Sliding-Window YOLO-OBB Detection (Off-Nadir Aligned + Exclusions) & Export GeoJSON
import cv2, io
from PIL import Image
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

TILE_URL = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'
WIN_PX = 1024
STRIDE_PX = 768
MIN_ASPECT = 0.65
MIN_DIAM_M = 5.0

session = requests.Session()
session.headers.update({'User-Agent': 'StorageMapper-Pilot/2.4'})

if 'tile_cache' not in globals():
    tile_cache = {}

def latlon_to_world_px(lat, lon, z):
    n = 2 ** z
    x = (lon + 180.0) / 360.0 * n * 256.0
    lat_r = math.radians(lat)
    y = (1.0 - math.asinh(math.tan(lat_r)) / math.pi) / 2.0 * n * 256.0
    return x, y

def world_px_to_latlon(px, py, z):
    n = 2 ** z
    lon = (px / 256.0) / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1.0 - 2.0 * (py / 256.0) / n))))
    return lat, lon

def fetch_tile(z, tx, ty):
    key = (z, tx, ty)
    if key in tile_cache:
        return tile_cache[key]
    for _ in range(2):
        try:
            r = session.get(TILE_URL.format(z=z, x=tx, y=ty), timeout=20)
            r.raise_for_status()
            arr = np.array(Image.open(io.BytesIO(r.content)).convert('RGB'))
            tile_cache[key] = arr
            time.sleep(0.03)
            return arr
        except Exception:
            time.sleep(0.3)
    return np.zeros((256, 256, 3), dtype=np.uint8)

def fetch_mosaic(cluster):
    lats = [t['lat'] for t in cluster]
    lons = [t['lon'] for t in cluster]
    dlat = BUFFER_M / 111320.0
    dlon = BUFFER_M / (111320.0 * math.cos(math.radians(float(np.mean(lats)))))
    x0, y1 = latlon_to_world_px(min(lats) - dlat, min(lons) - dlon, ZOOM)
    x1, y0 = latlon_to_world_px(max(lats) + dlat, max(lons) + dlon, ZOOM)

    tx0, tx1 = int(x0 // 256), int(x1 // 256)
    ty0, ty1 = int(y0 // 256), int(y1 // 256)
    nx, ny = tx1 - tx0 + 1, ty1 - ty0 + 1
    if nx * ny > MAX_TILES_PER_FARM:
        print(f'Skipping oversized cluster ({nx}x{ny}={nx*ny} tiles > {MAX_TILES_PER_FARM})')
        return None, 0, 0

    mosaic = np.zeros((ny * 256, nx * 256, 3), dtype=np.uint8)
    for ty in range(ty0, ty1 + 1):
        for tx in range(tx0, tx1 + 1):
            py, px = (ty - ty0) * 256, (tx - tx0) * 256
            mosaic[py:py+256, px:px+256] = fetch_tile(ZOOM, tx, ty)
    return mosaic, tx0 * 256, ty0 * 256

def window_starts(total, win, stride):
    if total <= win:
        return [0]
    starts = list(range(0, total - win + 1, stride))
    if starts[-1] != total - win:
        starts.append(total - win)
    return starts

def pt_seg_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx * dx + dy * dy
    if l2 < 1e-9:
        return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1) * dx + (py - y1) * dy) / l2))
    return math.hypot(px - (x1 + t * dx), py - (y1 + t * dy))

def match_with_off_nadir(known_all, dedup, m_per_px):
    if not known_all or not dedup:
        return set(), list(dedup), (0.0, 0.0)

    max_shift_px = 28.0 / m_per_px
    inlier_tol_px = 5.0 / m_per_px

    # 1) Estimate dominant base->roof off-nadir shift vector across the farm
    pairs = []
    for _, cx, cy, rpx in dedup:
        for ki, (ex, ey, er) in enumerate(known_all):
            dx, dy = cx - ex, cy - ey
            if math.hypot(dx, dy) <= max(max_shift_px, er * 3.0):
                ratio = rpx / max(er, 1e-6)
                if 0.55 <= ratio <= 1.85:
                    pairs.append((ki, dx, dy))

    shift_x, shift_y = 0.0, 0.0
    if len(pairs) >= 2:
        k_ids = np.array([p[0] for p in pairs])
        vecs = np.array([[p[1], p[2]] for p in pairs], dtype=np.float64)
        hyps = np.vstack([np.array([[0.0, 0.0]]), vecs])
        best_score, best_vecs = -1e9, None
        for hx, hy in hyps:
            dists = np.hypot(vecs[:, 0] - hx, vecs[:, 1] - hy)
            mask = dists <= inlier_tol_px
            unique_k = len(np.unique(k_ids[mask]))
            score = unique_k - 1e-4 * math.hypot(hx, hy)
            if score > best_score:
                best_score = score
                best_vecs = vecs[mask]
        if best_vecs is not None and len(best_vecs) >= 2:
            shift_x = float(np.median(best_vecs[:, 0]))
            shift_y = float(np.median(best_vecs[:, 1]))

    # 2) 1-to-1 Hungarian matching (shifted roof + same-azimuth height variation)
    L = math.hypot(shift_x, shift_y)
    ux, uy = (shift_x / L, shift_y / L) if L > 1e-3 else (0.0, 0.0)
    n_k, n_d = len(known_all), len(dedup)
    INF = 1e9
    cost = np.full((n_k, n_d), INF, dtype=np.float64)

    for ki, (ex, ey, er) in enumerate(known_all):
        sx, sy = ex + shift_x, ey + shift_y
        for di, (conf, cx, cy, rpx) in enumerate(dedup):
            d_shifted = math.hypot(cx - sx, cy - sy)
            base_tol = max(4.5 / m_per_px, er * 0.95, rpx * 0.95, 0.65 * (er + rpx))

            valid, c_val = False, d_shifted
            if d_shifted <= base_tol:
                valid = True
            elif L >= 3.0 / m_per_px:
                vx, vy = cx - ex, cy - ey
                proj = vx * ux + vy * uy
                perp = abs(-vx * uy + vy * ux)
                perp_tol = max(4.0 / m_per_px, 0.85 * max(er, rpx))
                if perp <= perp_tol and (-0.3 * er <= proj <= 1.85 * L + 0.5 * er):
                    ratio = rpx / max(er, 1e-6)
                    if 0.5 <= ratio <= 2.0:
                        valid = True
                        c_val = perp + 0.35 * abs(proj - L)
            else:
                d_raw = math.hypot(cx - ex, cy - ey)
                if d_raw <= base_tol:
                    valid = True
                    c_val = d_raw

            if valid:
                cost[ki, di] = c_val

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_known, matched_dets = set(), set()
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] < INF * 0.5:
            matched_known.add(r)
            matched_dets.add(c)

    # 3) Suppress double-detections sitting on a mapped tank's base / cylinder body
    unmatched = []
    for i in range(n_d):
        if i in matched_dets:
            continue
        _, cx, cy, rpx = dedup[i]
        on_mapped_body = any(
            L < 2.2 * er and pt_seg_dist(cx, cy, ex, ey, ex + shift_x, ey + shift_y) < max(er * 0.95, rpx * 0.75)
            for ex, ey, er in known_all
        )
        if not on_mapped_body:
            unmatched.append(dedup[i])

    return matched_known, unmatched, (shift_x, shift_y)

def scan_cluster_obb(cluster, cid):
    mosaic, ox, oy = fetch_mosaic(cluster)
    if mosaic is None:
        return None, [], [], [], 0, 0, (0.0, 0.0), 0.5
    H, W = mosaic.shape[:2]
    lat_mid = float(np.mean([t['lat'] for t in cluster]))
    m_per_px = 156543.03392 * math.cos(math.radians(lat_mid)) / (2 ** ZOOM)

    max_lat_m, min_lon_m = world_px_to_latlon(ox - 60, oy - 60, ZOOM)
    min_lat_m, max_lon_m = world_px_to_latlon(ox + W + 60, oy + H + 60, ZOOM)

    known_all = []
    for t in all_osm_tanks:
        if not (min_lat_m <= t['lat'] <= max_lat_m and min_lon_m <= t['lon'] <= max_lon_m):
            continue
        wx, wy = latlon_to_world_px(t['lat'], t['lon'], ZOOM)
        known_all.append((wx - ox, wy - oy, (t['diam_m'] / 2.0) / m_per_px))

    local_excl_polys, local_excl_nodes = [], []
    for ex in excluded_osm:
        if ex['max_lat'] < min_lat_m or ex['min_lat'] > max_lat_m or ex['max_lon'] < min_lon_m or ex['min_lon'] > max_lon_m:
            continue
        if ex['kind'] == 'node':
            wx, wy = latlon_to_world_px(ex['lat'], ex['lon'], ZOOM)
            local_excl_nodes.append((wx - ox, wy - oy))
        else:
            pts = np.array(
                [[latlon_to_world_px(p['lat'], p['lon'], ZOOM)[0] - ox,
                  latlon_to_world_px(p['lat'], p['lon'], ZOOM)[1] - oy] for p in ex['geom']],
                dtype=np.float32,
            )
            local_excl_polys.append(pts)

    raw_dets = []
    for y0 in window_starts(H, WIN_PX, STRIDE_PX):
        for x0 in window_starts(W, WIN_PX, STRIDE_PX):
            crop = mosaic[y0:min(y0 + WIN_PX, H), x0:min(x0 + WIN_PX, W)]
            preds = model.predict(
                Image.fromarray(crop),
                imgsz=WIN_PX,
                conf=CONF_THRESH,
                classes=[TANK_CLS_ID],
                device=DEVICE,
                verbose=False,
            )
            obb = preds[0].obb
            if obb is None or len(obb) == 0:
                continue
            xywhr = obb.xywhr.cpu().numpy()
            confs = obb.conf.cpu().numpy()
            for (cx, cy, w, h, _), conf in zip(xywhr, confs):
                mx = max(w, h)
                if mx <= 0 or (min(w, h) / mx) < MIN_ASPECT:
                    continue
                r_px = 0.25 * (w + h)
                if (r_px * 2.0 * m_per_px) < MIN_DIAM_M:
                    continue
                raw_dets.append((float(conf), float(x0 + cx), float(y0 + cy), float(r_px)))

    raw_dets.sort(reverse=True, key=lambda d: d[0])
    dedup = []
    for conf, cx, cy, rpx in raw_dets:
        if any(math.hypot(cx - hx, cy - hy) < max(rpx, hr) * 0.85 for _, hx, hy, hr in dedup):
            continue
        dedup.append((conf, cx, cy, rpx))

    matched_known_ids, remaining_dets, (shift_x, shift_y) = match_with_off_nadir(known_all, dedup, m_per_px)

    excluded_hits = 0
    candidates = []
    edge_tol_px = 5.0 / m_per_px

    for conf, cx, cy, rpx in remaining_dets:
        bx, by = cx - shift_x, cy - shift_y
        is_excluded = any(
            cv2.pointPolygonTest(poly, (cx, cy), True) >= -max(edge_tol_px, rpx * 0.5)
            or cv2.pointPolygonTest(poly, (bx, by), True) >= -max(edge_tol_px, rpx * 0.5)
            for poly in local_excl_polys
        ) or any(
            min(math.hypot(cx - nx, cy - ny), math.hypot(bx - nx, by - ny)) <= max(15.0 / m_per_px, rpx * 1.1)
            for nx, ny in local_excl_nodes
        )
        if is_excluded:
            excluded_hits += 1
            continue

        lat, lon = world_px_to_latlon(ox + bx, oy + by, ZOOM)
        candidates.append({
            'cluster_id': cid,
            'score': round(conf, 3),
            'lat': lat,
            'lon': lon,
            'diam_m': round(rpx * 2.0 * m_per_px, 1),
            'cx': cx,
            'cy': cy,
            'r_px': rpx
        })

    return mosaic, known_all, local_excl_polys, candidates, len(matched_known_ids), excluded_hits, (shift_x, shift_y), m_per_px

def circle_poly(lat, lon, diam_m, n=20):
    r_m = diam_m / 2.0
    dlat = r_m / 111320.0
    dlon = r_m / (111320.0 * math.cos(math.radians(lat)))
    ring = [[round(lon + dlon * math.cos(2 * math.pi * i / n), 7),
             round(lat + dlat * math.sin(2 * math.pi * i / n), 7)] for i in range(n)]
    ring.append(ring[0])
    return ring

target_clusters = [c for c in clusters if MIN_FARM_TANKS <= len(c) <= MAX_FARM_TANKS]
if PILOT_LIMIT and PILOT_LIMIT > 0:
    target_clusters = target_clusters[:PILOT_LIMIT]

all_candidates = []
scanned_clusters = []
show_plots = (PILOT_LIMIT is not None and 0 < PILOT_LIMIT <= 15)

for cid, cl in enumerate(target_clusters):
    t0 = time.time()
    mosaic, known_all, excl_polys, cands, matched_cnt, excl_cnt, (sx, sy), m_px = scan_cluster_obb(cl, cid)
    if mosaic is None:
        continue
    scanned_clusters.append((cid, cl))
    all_candidates.extend(cands)
    c_lat = float(np.mean([t['lat'] for t in cl]))
    c_lon = float(np.mean([t['lon'] for t in cl]))
    recall_pct = (100.0 * matched_cnt / len(known_all)) if known_all else 0.0
    shift_m = math.hypot(sx, sy) * m_px
    print(f'Farm #{cid} ({c_lat:.4f}, {c_lon:.4f}) [{time.time()-t0:.1f}s] | OSM recall: {matched_cnt}/{len(known_all)} ({recall_pct:.0f}%) | Shift: {shift_m:.1f}m | Excluded: {excl_cnt} | NEW: {len(cands)}')

    if show_plots:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(mosaic)
        for poly in excl_polys:
            ax.add_patch(MplPolygon(poly, closed=True, fill=False, edgecolor='#38bdf8', lw=1.2, alpha=0.85))
        for ex, ey, er in known_all:
            ax.add_patch(plt.Circle((ex, ey), er, fill=False, edgecolor='#4ade80', lw=1.4))
            if shift_m >= 2.0:
                ax.add_patch(plt.Circle((ex + sx, ey + sy), er, fill=False, edgecolor='#86efac', lw=0.9, ls=':', alpha=0.75))
        for c in cands:
            ax.add_patch(plt.Circle((c['cx'], c['cy']), c['r_px'], fill=False, edgecolor='#ef4444', lw=2.2, ls='--'))
            lbl = f"{c['score']:.2f}"
            ax.text(c['cx'], c['cy'] - c['r_px'] - 4, lbl, color='yellow',
                    fontsize=8, ha='center', weight='bold', bbox=dict(fc='black', alpha=0.6, pad=1, lw=0))
        ax.set_title(f'Farm #{cid} | Recall {matched_cnt}/{len(known_all)} | Shift {shift_m:.1f}m | Excluded {excl_cnt} | Red=New ({len(cands)})')
        ax.axis('off')
        plt.show()

all_candidates.sort(key=lambda c: c['score'], reverse=True)
geojson = {
    'type': 'FeatureCollection',
    'features': [{
        'type': 'Feature',
        'geometry': {'type': 'Polygon', 'coordinates': [circle_poly(c['lat'], c['lon'], c['diam_m'])]},
        'properties': {
            'score': c['score'],
            'est_diam_m': c['diam_m'],
            'cluster_id': c['cluster_id'],
            'coords': f"{c['lat']:.5f}, {c['lon']:.5f}",
            'id_url': f"https://www.openstreetmap.org/edit#map=19/{c['lat']:.5f}/{c['lon']:.5f}",
            'josm_url': f"http://127.0.0.1:8111/load_and_zoom?left={c['lon']-0.002:.5f}&right={c['lon']+0.002:.5f}&top={c['lat']+0.002:.5f}&bottom={c['lat']-0.002:.5f}"
        }
    } for c in all_candidates]
}

out_path = 'missing_tanks_pilot.geojson'
with open(out_path, 'w') as f:
    json.dump(geojson, f)
print()
print(f'Saved {len(all_candidates)} ranked candidates to {out_path} (z{ZOOM} tiles cached in RAM: {len(tile_cache)})')

if DOWNLOAD_GEOJSON and len(all_candidates) > 0:
    from google.colab import files
    files.download(out_path)


In [ ]:
#@title 5. Clickable Review Table (iD Editor + JOSM) & Interactive Map
import folium
from IPython.display import display, HTML

if not all_candidates:
    print('No unmapped candidates remaining after off-nadir alignment and clarifier/silo exclusions!')
else:
    rows = []
    for i, c in enumerate(all_candidates[:30], 1):
        id_url = f"https://www.openstreetmap.org/edit#map=19/{c['lat']:.5f}/{c['lon']:.5f}"
        josm = f"http://127.0.0.1:8111/load_and_zoom?left={c['lon']-0.002:.5f}&right={c['lon']+0.002:.5f}&top={c['lat']+0.002:.5f}&bottom={c['lat']-0.002:.5f}"
        rows.append(
            f"<tr><td style='padding:4px 8px'>{i}</td><td style='padding:4px 8px'>Farm #{c['cluster_id']}</td>"
            f"<td style='padding:4px 8px'><b>{c['score']:.3f}</b></td><td style='padding:4px 8px'>~{c['diam_m']:.1f}m</td>"
            f"<td style='padding:4px 8px'><code>{c['lat']:.5f}, {c['lon']:.5f}</code></td>"
            f"<td style='padding:4px 8px'><a href='{id_url}' target='_blank'>Open in iD</a> · <a href='{josm}' target='_blank'>JOSM</a></td></tr>"
        )
    table_html = (
        "<table style='border-collapse:collapse;font-family:monospace;font-size:12px'>"
        "<tr><th style='padding:4px 8px'>#</th><th style='padding:4px 8px'>Farm</th>"
        "<th style='padding:4px 8px'>Conf</th><th style='padding:4px 8px'>Diam</th>"
        "<th style='padding:4px 8px'>Coords</th><th style='padding:4px 8px'>Links</th></tr>"
        + ''.join(rows) + '</table>'
    )
    display(HTML(table_html))

    top = all_candidates[0]
    m = folium.Map(location=[top['lat'], top['lon']], zoom_start=17, tiles=None, max_zoom=19)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri World Imagery', name='Esri Satellite', max_native_zoom=18, max_zoom=19
    ).add_to(m)
    for cid, cl in scanned_clusters:
        for t in cl:
            folium.Polygon(
                locations=[[p['lat'], p['lon']] for p in t['geom']],
                color='#4ade80', weight=1.5, fill=True, fill_opacity=0.2,
                tooltip=f"OSM Tank #{t['id']} (~{t['diam_m']:.0f}m)"
            ).add_to(m)
    for c in all_candidates:
        pts = [[lat, lon] for lon, lat in circle_poly(c['lat'], c['lon'], c['diam_m'])]
        folium.Polygon(
            locations=pts, color='#ef4444', weight=2.5, dash_array='5,5',
            fill=True, fill_opacity=0.15,
            tooltip=f"Conf {c['score']:.2f} (~{c['diam_m']:.0f}m)"
        ).add_to(m)
    m.save('pilot_map.html')
    display(HTML(m._repr_html_()))
